# 날짜별 훈련 추천 결과 조회

4장에서 만든 규칙 기반 추천 결과를 불러와, 특정 날짜의 훈련 맥락과 검토용 추천 문장을 확인한다.

이 조회 결과는 추천일 이전 기록으로 만든 규칙 기반 안내이며, 실제 피로·회복 상태나 개별 훈련 처방을 확정하지 않는다.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

recommendation_result_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_rule_based_recommendations.csv"
)

recommendation_result_df = pd.read_csv(
    recommendation_result_path,
    parse_dates=["recommendation_date"],
)

recommendation_result_df.shape

(100, 15)

In [2]:
recommendation_lookup_df = (
    recommendation_result_df
    .set_index("recommendation_date")
)

recommendation_lookup_columns = [
    "relative_recent_load_level",
    "short_to_long_tss_ratio",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_record_status",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
    "recommendation_context_summary",
    "recommendation_guidance",
]

recommendation_date = "2009-06-02"

recommendation_lookup_df.loc[
    [recommendation_date],
    recommendation_lookup_columns,
].T

recommendation_date,2009-06-02 00:00:00+00:00
relative_recent_load_level,relatively_high
short_to_long_tss_ratio,1.21454
previous_7_day_workout_hours,13.3325
previous_7_day_ride_count,8.0
previous_day_record_status,recorded
previous_day_recorded_ride_streak_days,12.0
previous_day_no_record_streak_days,0.0
recommendation_context_summary,최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.21입니다....
recommendation_guidance,"최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다."


In [3]:
def format_recommendation(recommendation_date):
    recommendation_row = recommendation_lookup_df.loc[
        recommendation_date
    ]

    return "\n".join(
        [
            f"추천일: {recommendation_date}",
            (
                "훈련 맥락: "
                f"{recommendation_row['recommendation_context_summary']}"
            ),
            (
                "추천 안내: "
                f"{recommendation_row['recommendation_guidance']}"
            ),
        ]
    )

print(
    format_recommendation("2009-06-02")
)

추천일: 2009-06-02
훈련 맥락: 최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.21입니다. 전날에 라이드 기록이 있으며, 연속 라이드 기록은 12일입니다.
추천 안내: 최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다.


In [4]:
print(
    format_recommendation("2009-05-09")
)

추천일: 2009-05-09
훈련 맥락: 최근 부하가 과거 흐름보다 낮은 편입니다. 최근 7일 TSS 비율은 0.77입니다. 전날에는 라이드 기록이 없으며, 연속 무기록은 2일입니다.
추천 안내: 최근 부하가 낮고 전날 기록도 없습니다. 기록 맥락을 확인한 뒤, 점진적인 훈련 재개 가능성을 검토합니다.


In [5]:
def format_recommendation(recommendation_date):
    try:
        recommendation_row = recommendation_lookup_df.loc[
            recommendation_date
        ]
    except KeyError:
        return (
            f"추천일 {recommendation_date}의 "
            "추천 결과를 찾을 수 없습니다."
        )

    return "\n".join(
        [
            f"추천일: {recommendation_date}",
            (
                "훈련 맥락: "
                f"{recommendation_row['recommendation_context_summary']}"
            ),
            (
                "추천 안내: "
                f"{recommendation_row['recommendation_guidance']}"
            ),
        ]
    )

print(
    format_recommendation("2009-01-01")
)

추천일 2009-01-01의 추천 결과를 찾을 수 없습니다.


In [6]:
print(
    format_recommendation("2009-06-02")
)

추천일: 2009-06-02
훈련 맥락: 최근 부하가 과거 흐름보다 높은 편입니다. 최근 7일 TSS 비율은 1.21입니다. 전날에 라이드 기록이 있으며, 연속 라이드 기록은 12일입니다.
추천 안내: 최근 부하와 전날 기록이 이어져 있어, 다음 훈련 강도는 보수적으로 검토합니다.


### 날짜별 추천 결과 조회 결과

- 4장에서 저장한 100행 15열의 규칙 기반 추천 결과를 불러와, 날짜를 인덱스로 사용하는 조회용 표를 만들었다.
- `format_recommendation()` 함수에 날짜를 입력하면 훈련 맥락과 검토용 추천 안내를 세 줄의 사용자용 문장으로 반환한다.
- 높은 부하·전날 기록 사례인 2009-06-02와 낮은 부하·전날 무기록 사례인 2009-05-09를 조회해, 서로 다른 규칙의 안내가 정상적으로 표시되는 것을 확인했다.
- 저장된 범위 밖 날짜는 오류를 그대로 표시하지 않고, 추천 결과가 없다는 안내 문장으로 처리한다. 이 조회 결과는 이전 기록에 기반한 규칙형 안내이며 실제 피로·회복 상태나 개별 훈련 처방을 확정하지 않는다.